# LLDA trie_beams: 2.000 artigos, 54 classes e execução em GPU

Expandimos o ensaio de cinco artigos para o conjunto completo. A LLM recebe o resumo e o catálogo fixo de **54 nomes legíveis**; uma árvore de prefixos permite gerar somente esses nomes. Cada documento retorna quatro hipóteses de beam search.

**O que mudou:** resumos completos, todas as classes, processamento em lotes CUDA e checkpoint após cada lote. Os códigos originais são preservados. Classes próximas recebem qualificadores, por exemplo **Information Theory (Computer Science)** e **Information Theory (Mathematics)**, para manter 54 saídas distintas.

Os rótulos reais servem para conferir a recuperação dos textos e avaliar as respostas; não entram no prompt nem na escolha dos tokens. LLDA aqui é o método por LLM do projeto, não Labeled LDA.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd
import torch
from IPython.display import display
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'dataset_loader.py').exists())
EXPERIMENT_DIR = ROOT / 'experimento_classificacao_topicos'
sys.path.insert(0, str(EXPERIMENT_DIR))
from llda_trie_54 import CLASSES, LABELS, full_dataset, require_gpu, run_gpu

pd.set_option('display.max_colwidth', 100)
df = full_dataset()
print(f'{len(df)} documentos completos; {len(CLASSES)} classes; IDs originais preservados')
display(pd.DataFrame({'Classe': CLASSES, 'Nome para LLM': LABELS}))
display(df[['doc_id', 'fonte_texto', 'classes_reais_aapd']].head())
print('Caracteres por resumo:', df.resumo_texto.str.len().describe().round(1).to_dict())

2000 documentos completos; 54 classes; IDs originais preservados
Caracteres por resumo: {'count': 2000.0, 'mean': 985.1, 'std': 394.3, 'min': 37.0, '25%': 703.8, '50%': 951.0, '75%': 1235.0, 'max': 3132.0}


,Classe,Nome para LLM
0,cmp-lg,Computation and Language (Legacy Category)
1,cond-mat.dis-nn,Disordered systems and neural networks
2,cond-mat.stat-mech,Statistical mechanics
3,cs.AI,Artificial intelligence
4,cs.CC,Computational complexity
5,cs.CE,Computational engineering finance and science
6,cs.CG,Computational geometry
7,cs.CL,Computational linguistics and natural language processing
8,cs.CR,Cryptography and security
9,cs.CV,Computer vision and pattern recognition


,doc_id,fonte_texto,classes_reais_aapd
0,51,test.txt:artigo_51,"math.CO, cs.DM"
1,178,test.txt:artigo_178,"physics.soc-ph, cs.DL"
2,209,test.txt:artigo_209,"cs.SI, physics.soc-ph"
3,228,test.txt:artigo_228,"cs.IT, math.IT"
4,285,test.txt:artigo_285,"cs.DC, cs.DS"


## 1. Configurar a GPU e o decoding

Mantemos **Qwen2.5-0.5B-Instruct** para comparar com o ensaio anterior. A GPU acelera lotes de oito documentos, cada um com quatro beams. O código exige CUDA e não muda silenciosamente para CPU.

O score do beam usa penalização por comprimento; não representa probabilidade calibrada de uma classe. Os beams são hipóteses de sequência, não amostras independentes. Usamos os três primeiros para a avaliação multi-rótulo, sem consultar quantas classes reais cada artigo possui.

In [2]:
MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
BATCH_SIZE = 8
NUM_BEAMS = 4
TOP_K = 3
LOCAL_FILES_ONLY = False
CACHE_DIR = None  # opcional: cache do modelo
OUTPUT_DIR = EXPERIMENT_DIR / 'resultados' / 'llda_trie_54_gpu'

gpu = require_gpu()
print(f"GPU: {gpu['name']} | VRAM livre: {gpu['free_bytes'] / 1024**3:.2f} GiB")

GPU: NVIDIA GeForce RTX 5080 | VRAM livre: 14.65 GiB


## 2. Executar com restrição por trie

O callback `prefix_allowed_tokens_fn` consulta a árvore após cada token. EOS só é permitido ao completar um nome. O limite de geração cobre o nome tokenizado mais longo e o EOS. Todas as saídas são validadas por igualdade exata.

Agrupamos textos de comprimentos próximos para reduzir padding e usamos padding à esquerda, apropriado para geração causal em lotes. O checkpoint é substituído atomicamente após cada lote. Ao repetir a célula, documentos concluídos são reutilizados; parâmetros ou dados diferentes exigem outra pasta de resultados.

O CSV comparativo guarda apenas o começo dos resumos. Recuperamos os textos completos nos arquivos locais AAPD por prefixo e classes; não truncamos os artigos para a geração.

In [3]:
df, outputs, metrics, per_class = run_gpu(
    output_dir=OUTPUT_DIR, model_name=MODEL_NAME, batch_size=BATCH_SIZE,
    num_beams=NUM_BEAMS, top_k=TOP_K, cache_dir=CACHE_DIR,
    local_files_only=LOCAL_FILES_ONLY,
)
display(metrics.round(4))

GPU: NVIDIA GeForce RTX 5080; 2000 artigos; 2000 concluídos; 0 pendentes
 documentos  classes  validade_exata  acerto_top1  precisao_top3  recall_top3  F1 micro  F1 macro (54)  Jaccard
       2000       54             1.0       0.1595         0.1608       0.1792    0.1694         0.1177   0.1124


,documentos,classes,validade_exata,acerto_top1,precisao_top3,recall_top3,F1 micro,F1 macro (54),Jaccard
0,2000,54,1.0,0.1595,0.1608,0.1792,0.1694,0.1177,0.1124


## 3. Inspecionar as atribuições

Cada beam produz um nome legível e seu código de classe. Conformidade com o catálogo não garante acerto semântico. Pares de categorias equivalentes ou coocorrentes, como `cs.IT`/`math.IT`, continuam difíceis de distinguir pelo conteúdo.

In [4]:
display(outputs.merge(df[['doc_id', 'classes_reais_aapd']], on='doc_id', validate='many_to_one').head(24))
print('Documentos concluídos:', outputs.doc_id.nunique())
print('Nomes fora do catálogo:', (~outputs.topico.isin(LABELS)).sum())
execution_file = OUTPUT_DIR / 'execucao_gpu.json'
if execution_file.exists():
    display(pd.DataFrame([json.loads(execution_file.read_text())]))

Documentos concluídos: 2000
Nomes fora do catálogo: 0


,doc_id,rank,topico,classe,score_beam,valido,classes_reais_aapd
0,7,1,Statistics Theory (Statistics),stat.TH,-4.724256e-01,True,"cs.GT, cs.NI"
1,7,2,Computer science and game theory,cs.GT,-6.137755e-01,True,"cs.GT, cs.NI"
2,7,3,Statistics Theory (Mathematics),math.ST,-8.586167e-01,True,"cs.GT, cs.NI"
3,7,4,Statistics methodology,stat.ME,-1.137666e+00,True,"cs.GT, cs.NI"
4,1754,1,Statistical machine learning,stat.ML,-6.822087e-01,True,"cs.AI, cs.NE"
5,1754,2,Computer science and game theory,cs.GT,-9.626899e-01,True,"cs.AI, cs.NE"
6,1754,3,Statistical mechanics,cond-mat.stat-mech,-1.458412e+00,True,"cs.AI, cs.NE"
7,1754,4,Statistical machine learning,stat.ML,-1.666667e+08,True,"cs.AI, cs.NE"
8,259,1,Statistical machine learning,stat.ML,-7.538013e-01,True,"cs.IT, math.IT"
9,259,2,Statistics Theory (Statistics),stat.TH,-1.016338e+00,True,"cs.IT, math.IT"


,gpu,segundos_execucao,artigos_nesta_execucao,vram_pico_gib
0,NVIDIA GeForce RTX 5080,68.332976,2000,2.305996


## 4. Recuperação das classes originais

Acerto top-1 aceita qualquer classe verdadeira. Precisão e recall top-3 comparam os três primeiros nomes distintos com o conjunto de rótulos reais; F1 micro/macro e Jaccard avaliam as mesmas atribuições. O F1 macro mantém as 54 classes, inclusive as raras.

Não calculamos LRAP do ranking completo: quatro beams não fornecem os scores de todas as 54 classes. Os resultados descrevem a classificação desses 2.000 artigos; não houve treinamento nem ajuste de parâmetros usando seus rótulos.

In [5]:
display(per_class.sort_values('F1', ascending=False))
px.bar(per_class, x='Classe', y='F1', hover_data=['Nome', 'Suporte', 'Precisão', 'Recall'],
       title='LLDA trie_beams — F1 por classe original (top-3)', height=550).show()
px.scatter(per_class, x='Suporte', y='F1', hover_name='Classe', hover_data=['Nome'],
           title='Suporte da classe e qualidade das atribuições').show()

,Classe,Nome,Precisão,Recall,F1,Suporte
21,cs.LG,Machine learning,0.672515,0.391156,0.494624,294
22,cs.LO,Logic in computer science,0.694915,0.310606,0.429319,132
40,math.NT,Number theory,0.500000,0.333333,0.400000,9
49,quant-ph,Quantum physics,0.653846,0.278689,0.390805,61
34,cs.SI,Social and information networks,0.775000,0.231343,0.356322,134
52,stat.ML,Statistical machine learning,0.191919,0.821622,0.311157,185
20,cs.IT,Information Theory (Computer Science),0.928571,0.152269,0.261635,683
19,cs.IR,Information retrieval,0.451613,0.181818,0.259259,77
41,math.OC,Optimization and control,0.297297,0.215686,0.250000,102
3,cs.AI,Artificial intelligence,0.327273,0.170616,0.224299,211


Gráfico interativo: comparação dos métodos

Gráfico interativo: comparação dos métodos

## 5. Arquivos e próximas comparações

A pasta de resultados contém textos completos e suas fontes, catálogo, outputs, métricas gerais e por classe, configuração, revisão do modelo, tempo de execução e pico de VRAM. O notebook usa checkpoints para permitir retomada sem repetir todo o corpus.

Próximas comparações: variar o número de beams, testar um modelo maior e comparar com Seeded LDA/Guided BERTopic usando os mesmos **textos completos**. Os experimentos anteriores que leram diretamente `resumo_texto` do CSV usaram fragmentos e não são uma comparação equivalente de entrada.

In [6]:
print(f'Resultados: {OUTPUT_DIR}')
display(pd.DataFrame({'Arquivo': sorted(p.name for p in OUTPUT_DIR.iterdir() if p.is_file())}))

Resultados: /home/jesus/Documentos/LLDA/experimento_classificacao_topicos/resultados/llda_trie_54_gpu


,Arquivo
0,artigos_completos.csv
1,catalogo.csv
2,configuracao.json
3,execucao_gpu.json
4,metricas.csv
5,metricas_por_classe.csv
6,outputs.csv


## Resultado da execução na RTX 5080

Com Qwen2.5-0.5B-Instruct, lotes de 8 e quatro beams por documento:

- **2.000 artigos concluídos**, com 8.000 hipóteses e validade exata de 100%.
- **Acerto top-1: 15,95%**; F1 micro top-3: 0,1694; F1 macro: 0,1177.
- Inferência e checkpoint dos lotes: **68,33 segundos**, excluindo preparação/carregamento; pico de memória CUDA alocada: **2,31 GiB**.

A restrição funcionou para todas as respostas. A baixa recuperação das classes mostra que controlar o formato não resolve sozinho a classificação; a próxima comparação pode usar um modelo maior e revisar os nomes das categorias.
